# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
rows = [] # crea una lista vacia donde vamos a guardar todas las filas del CSV
row = {} #crea un diccionario vacio. Luego esta variable se usará para guardar solo la primera fila. 

with DATA_PATH.open(newline="", encoding="utf-8") as f: #abre el archivo que esta en datapath
    reader = csv.DictReader(f) #leo el csv con dictreader
    rows = list(reader) #convierte todas las filas que ha leido reader en una lista y los guarda en rows

print(len(rows))
print(reader.fieldnames)

row = rows[0]




5
['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? 
sample_id

2. ¿Cuántas características numéricas hay?
hay 11 características: fixed_acidity, volatile_acidity, citric_acid, residual_sugar, chlorides, free_sulfur_dioxide, total_sulfur_dioxide, density, ph, sulphates, alcohol

3. ¿Qué columna extra añadirías para probar el contrato?
extra_column, porque la idea es comprobar si el contrato rechaza campos no definidos

4. ¿Qué valor de `ph` sería claramente inválido?
-1 porque un ph negativo no tendria sentido para este contexto

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [4]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    # TODO: declara las 11 características con Field(ge=..., le=...).
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

model_row = row.copy()
model_row.pop("sample_id")

request = WineQualityRequestPydantic.model_validate(model_row) #le pasa esa fila a pydantic para comprobar que todo esta bien

print(request)

# TODO: separa sample_id de la fila y valida solo las características del modelo.
# model_row = ...
# request = WineQualityRequestPydantic.model_validate(model_row)

fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [5]:
# TODO: completa los casos y captura ValidationError.
# Caso 1: fila válida
try:
    request = WineQualityRequestPydantic.model_validate(model_row)
    print(request)
except ValidationError as error:
    print(error)

# Caso 2: columna inesperada
row_extra = model_row.copy()
row_extra["unexpected_field"] = 123

try:
    request = WineQualityRequestPydantic.model_validate(row_extra)
    print(request)
except ValidationError as error:
    print(error)

# Caso 3: ph fuera de rango
row_bad_ph = model_row.copy()
row_bad_ph["ph"] = 10

try:
    request = WineQualityRequestPydantic.model_validate(row_bad_ph)
    print(request)
except ValidationError as error:
    print(error)

fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4
1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [6]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.
valid_request = WineQualityRequestDataclass(
    fixed_acidity=7.4,
    volatile_acidity=0.7,
    citric_acid=0.0,
    residual_sugar=1.9,
    chlorides=0.076,
    free_sulfur_dioxide=11,
    total_sulfur_dioxide=34,
    density=0.9978,
    ph=3.51,
    sulphates=0.56,
    alcohol=9.4
)

print(valid_request)


bad_request = WineQualityRequestDataclass(
    fixed_acidity=7.4,
    volatile_acidity=0.7,
    citric_acid=0.0,
    residual_sugar=1.9,
    chlorides=0.076,
    free_sulfur_dioxide=11,
    total_sulfur_dioxide=34,
    density=0.9978,
    ph=99.0,
    sulphates=0.56,
    alcohol=9.4
)

print(bad_request)

WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11, total_sulfur_dioxide=34, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4)
WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11, total_sulfur_dioxide=34, density=0.9978, ph=99.0, sulphates=0.56, alcohol=9.4)


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en la entrada de datos porque valida automaticamente tipos, rangos y campos extra. Usaría `dataclass` en estructuras internas porque  porque son mas simples y ligeras cuando los datos porque son mas simples y ligeras cuando los datos ya estan validados.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [8]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

# TODO: construye vector usando getattr(request, name).
vector = [getattr(request, name) for name in FEATURE_NAMES] #recorre cada nombre de feature_names y obtiene ese atributo de request

### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores? Sí. Aunque intercambies density y alcohol, sigue habiendo 11 valores.

- ¿El clasificador detectaría necesariamente el error?No. Para el modelo seguiría siendo una lista de 11 números, así que puede aceptar el vector sin saber que dos características están cambiadas de posición. El problema es que haría la predicción con datos mal colocados.

- ¿Cómo evitarías que cada script definiera un orden distinto?
Definiendo FEATURE_NAMES una sola vez en un sitio común y reutilizándolo en todos los scripts.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [9]:
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    sample_id: str
    predicted_class: str
    confidence: float = Field(ge=0, le=1)
    model_version: str
    preprocessing_version: str

# TODO: escribe un ejemplo de salida para red-001.
prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    predicted_class="acceptable",
    confidence=0.87,
    model_version="v1",
    preprocessing_version="v1"
)

print(prediction)


sample_id='red-001' predicted_class='acceptable' confidence=0.87 model_version='v1' preprocessing_version='v1'


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

Plan de implementación

contracts.py
Contendrá los contratos de entrada y salida con Pydantic.
Aquí se definirán WineQualityRequestPydantic y WineQualityPredictionPydantic,
incluyendo los tipos, rangos permitidos y la prohibición de campos extra.

preprocess.py
Contendrá el orden fijo de las 11 características mediante FEATURE_NAMES.
También transformará una petición validada en el vector numérico que espera
el modelo.

inference.py
Se encargará de recibir los datos ya validados y preprocesados, ejecutar
el modelo y construir la salida con la clase predicha, la confianza y las
versiones utilizadas.

predict_file.py
Leerá el CSV, separará sample_id de las características, validará cada fila,
llamará al preprocesado y a la inferencia, y mostrará o guardará las
predicciones.